# Consistency & flow-matching models — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normal_pdf(x,mu,sigma):
    return np.exp(-0.5*((x-mu)/sigma)**2)/(sigma*np.sqrt(2*np.pi))
def softmax(a):
    a=np.asarray(a,dtype=float); e=np.exp(a-a.max()); return e/e.sum()
def mse(a,b): return float(np.mean((np.asarray(a)-np.asarray(b))**2))
print('setup ok')

## learned transport paths
Diffusion (10.12) gives a slow reverse process; normalizing flows (10.9) give invertible transformations. Flow matching learns velocities along a path, while consistency models learn predictions that agree across time. Video and 3D generation (10.19) often need these faster paths.

In [ ]:
# 1) linear path between noise and data
x0=0.; x1=2.; t=np.linspace(0,1,80); xt=(1-t)*x0+t*x1
plt.figure(figsize=(5,3)); plt.plot(t,xt); plt.xlabel('t'); plt.ylabel('x_t'); plt.title('straight probability path')
assert abs(((1-.3)*x0+.3*x1)-0.6)<1e-12

In [ ]:
# 2) velocity target is constant on a linear path
v=np.ones_like(t)*(x1-x0)
plt.figure(figsize=(5,3)); plt.plot(t,v); plt.ylim(0,3); plt.title('flow-matching target velocity')
assert np.allclose(v,2)

In [ ]:
# 3) Euler steps follow the learned velocity
x=0.; path=[x]
for _ in range(10): x=x+0.1*2; path.append(x)
plt.figure(figsize=(5,3)); plt.plot(path,'o-'); plt.title('few large steps from noise to data')
assert abs(path[-1]-2)<1e-12

In [ ]:
# 4) curved path has time-varying velocity
t=np.linspace(0,1,80); xt=t**2; vt=2*t
plt.figure(figsize=(5,3)); plt.plot(t,xt,label='x_t'); plt.plot(t,vt,label='v_t'); plt.legend(); plt.title('path choice changes velocity')
assert vt[-1]>vt[10]

In [ ]:
# 5) consistency means predictions agree across time
pred_t0=np.array([2.0,2.0,2.0]); pred_t1=np.array([2.05,1.95,2.0]); err=np.abs(pred_t0-pred_t1)
plt.figure(figsize=(4,3)); plt.bar(range(3),err); plt.title('consistency error across times')
assert err.max()<0.1